# EdAi Project — Malaria Screening on a Raspberry Pi (Phase 1: training on Kaggle)

**Device goal:** a Raspberry Pi classifies a blood-smear cell image; **the LED turns ON when the cell is Parasitized.**

**Research question:** when the model is compressed (INT8) for the edge, does it stay clinically reliable
(sensitivity, calibration) — and what does it cost in system resources?

This notebook:
1. Writes the project modules to `src/` (one module per step)
2. Builds patient-level splits (no patient in both train and test)
3. Trains MobileNetV3-Small while logging **model metrics** and **system metrics**
4. Calibrates the model (temperature scaling) and picks the **LED threshold**
5. Exports to ONNX, quantizes to INT8, compares FP32 vs INT8
6. Saves everything the Pi needs to `outputs/deploy/`

**Before running:** Add Input → *Malaria Cell Images Dataset* (iarunava) · Accelerator → **GPU** · Internet → **On** (for ImageNet weights and onnxruntime).

## 0. Setup

In [ ]:
import os, sys, glob, json, shutil, subprocess

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working/EdAi_project" if ON_KAGGLE else os.path.abspath("EdAi_project_run")
for d in ("src", "splits", "outputs"):
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.chdir(WORK)

# onnx / onnxruntime are needed for export + quantization
try:
    import onnx, onnxruntime
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)

def find_data_root(base="/kaggle/input"):
    """Folder that contains Parasitized/ and Uninfected/ (shallowest copy)."""
    hits = sorted(glob.glob(os.path.join(base, "**", "Parasitized"), recursive=True),
                  key=lambda p: p.count(os.sep))
    if not hits:
        raise FileNotFoundError("Malaria dataset not found - add it with 'Add Input'.")
    return os.path.dirname(hits[0])

DATA_ROOT = find_data_root()
print("Working dir:", WORK)
print("Data root  :", DATA_ROOT)

## 1. Project modules
Each cell below writes one file of the project. The same files are in the GitHub repo.

### `src/config.py` — all settings in one place

In [ ]:
%%writefile src/config.py
"""
Central configuration for the EdAi malaria edge project.
Every other module imports its settings from here, so experiments
are controlled from one place.
"""
import os
from pathlib import Path

# ---- Paths -------------------------------------------------------------
ROOT = Path(__file__).resolve().parents[1]      # project root (EdAi_project/)
SPLITS_DIR = ROOT / "splits"                     # split CSVs (committed to git)
OUTPUT_DIR = ROOT / "outputs"                    # models, logs, plots (not committed)

# Default dataset location on Kaggle (folder that contains Parasitized/ and Uninfected/)
KAGGLE_DATA_ROOT = "/kaggle/input/cell-images-for-detecting-malaria/cell_images"

# ---- Data --------------------------------------------------------------
# Index 0 = negative, index 1 = positive (what the red LED means)
CLASSES = ["Uninfected", "Parasitized"]
IMG_SIZE = 128                       # cells are small (~100-150 px); 128 keeps the Pi fast
SPLIT_RATIOS = (0.70, 0.15, 0.15)    # train / val / test

# ---- Reproducibility ---------------------------------------------------
SEED = 42

# ---- Model + training (Step 3-4) ---------------------------------------
ARCH = "mobilenet_v3_small"          # also: "mobilenet_v2", "resnet18"
PRETRAINED = True                    # ImageNet weights (needs internet on Kaggle)
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = min(4, os.cpu_count() or 2)

# ---- LED decision rule (Step 6) ----------------------------------------
# The LED turns ON when P(Parasitized) >= threshold.
# The threshold is chosen on the VALIDATION set as the highest value that
# still catches at least TARGET_SENSITIVITY of infected cells
# (missing an infection is worse than a false alarm).
TARGET_SENSITIVITY = 0.98

# ---- Quantization (Step 8) ---------------------------------------------
CALIB_IMAGES = 300                   # training images used to calibrate INT8 ranges

### `src/data_prep.py` — Step 1: patient IDs, de-duplication, splits

In [ ]:
%%writefile src/data_prep.py
"""
STEP 1 - DATA PREPARATION
=========================
Builds the image list for the NIH malaria cell dataset and creates
train/val/test splits.

What this module does
---------------------
1. Finds every cell image under the dataset folder.
2. Removes duplicates. (The Kaggle copy contains the whole dataset twice:
   cell_images/ and cell_images/cell_images/.)
3. Extracts the PATIENT ID from each filename.
   Example: "C33P1thinF_IMG_..._cell_179.png" -> patient "C33"
   The dataset has 200 patients (150 infected, 50 uninfected).
4. Makes TWO splits so we can compare them in the report:
   - patient_split.csv : no patient appears in more than one split (honest)
   - random_split.csv  : images shuffled randomly (what most tutorials do;
                          cells of one patient leak into train AND test)
5. Prints statistics and a leakage check.

Output: splits/manifest.csv, splits/patient_split.csv, splits/random_split.csv
Paths in the CSVs are RELATIVE to the data root, so the same CSV works on
Kaggle, on a laptop and on the Raspberry Pi.

Usage
-----
Kaggle : python src/data_prep.py --data_root /kaggle/input/cell-images-for-detecting-malaria/cell_images
Local  : python src/data_prep.py --data_root data/cell_images
Add --verify to open every image and drop corrupted files (slower).
"""
import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd

from config import CLASSES, SEED, SPLIT_RATIOS, SPLITS_DIR

PATIENT_RE = re.compile(r"^C(\d+)")   # every filename starts with C<number>


# ----------------------------------------------------------------------
# 1. Collect images
# ----------------------------------------------------------------------
def collect_images(data_root: Path, verify: bool = False) -> pd.DataFrame:
    """Return one row per unique image: path, class name, label, patient_id."""
    rows, seen = [], set()

    for label, cls in enumerate(CLASSES):
        # All folders named like the class (there may be a duplicated nested copy).
        # Sort by depth so the shallowest copy wins and duplicates are skipped.
        class_dirs = sorted((d for d in data_root.rglob(cls) if d.is_dir()),
                            key=lambda d: len(d.parts))
        if not class_dirs:
            raise FileNotFoundError(f"No '{cls}' folder found under {data_root}")

        for d in class_dirs:
            for img in sorted(d.glob("*.png")):
                key = (cls, img.name)
                if key in seen:                 # duplicate copy -> skip
                    continue
                seen.add(key)

                m = PATIENT_RE.match(img.name)
                if m is None:
                    print(f"[warn] cannot read patient id from {img.name}, skipped")
                    continue

                rows.append({
                    "path": img.relative_to(data_root).as_posix(),
                    "class": cls,
                    "label": label,
                    "patient_id": f"C{m.group(1)}",
                })

    df = pd.DataFrame(rows)

    if verify:
        df = drop_corrupted(df, data_root)
    return df


def drop_corrupted(df: pd.DataFrame, data_root: Path) -> pd.DataFrame:
    """Open every image once; drop files that cannot be decoded."""
    from PIL import Image
    bad = []
    for i, p in enumerate(df["path"]):
        try:
            with Image.open(data_root / p) as im:
                im.verify()
        except Exception:
            bad.append(i)
    if bad:
        print(f"[info] dropped {len(bad)} corrupted images")
    return df.drop(index=bad).reset_index(drop=True)


# ----------------------------------------------------------------------
# 2. Splits
# ----------------------------------------------------------------------
def _cut(n: int, ratios) -> tuple:
    """Number of items for train and val (test gets the rest)."""
    n_train = int(round(n * ratios[0]))
    n_val = int(round(n * ratios[1]))
    return n_train, n_val


def patient_level_split(df: pd.DataFrame, ratios=SPLIT_RATIOS, seed=SEED) -> pd.DataFrame:
    """
    Split by PATIENT, not by image.
    Patients are stratified into two groups (infected / uninfected patients)
    so every split gets both kinds.
    """
    rng = np.random.default_rng(seed)
    infected = df.groupby("patient_id")["label"].max()   # 1 if patient has any parasitized cell

    assignment = {}
    for _, group in infected.groupby(infected):
        ids = np.array(sorted(group.index))
        rng.shuffle(ids)
        n_tr, n_va = _cut(len(ids), ratios)
        for pid in ids[:n_tr]:
            assignment[pid] = "train"
        for pid in ids[n_tr:n_tr + n_va]:
            assignment[pid] = "val"
        for pid in ids[n_tr + n_va:]:
            assignment[pid] = "test"

    out = df.copy()
    out["split"] = out["patient_id"].map(assignment)
    return out


def random_image_split(df: pd.DataFrame, ratios=SPLIT_RATIOS, seed=SEED) -> pd.DataFrame:
    """Common tutorial split: shuffle images (stratified by class), ignore patients."""
    rng = np.random.default_rng(seed)
    out = df.copy()
    out["split"] = ""
    for _, idx in out.groupby("label").groups.items():
        idx = np.array(idx)
        rng.shuffle(idx)
        n_tr, n_va = _cut(len(idx), ratios)
        out.loc[idx[:n_tr], "split"] = "train"
        out.loc[idx[n_tr:n_tr + n_va], "split"] = "val"
        out.loc[idx[n_tr + n_va:], "split"] = "test"
    return out


# ----------------------------------------------------------------------
# 3. Reporting
# ----------------------------------------------------------------------
def summarize(df: pd.DataFrame, name: str) -> None:
    print(f"\n=== {name} ===")
    table = pd.crosstab(df["split"], df["class"], margins=True)
    table["patients"] = df.groupby("split")["patient_id"].nunique()
    table.loc["All", "patients"] = df["patient_id"].nunique()
    print(table.reindex(["train", "val", "test", "All"]))

    # Leakage check: patients appearing in more than one split
    train_p = set(df.loc[df.split == "train", "patient_id"])
    test_p = set(df.loc[df.split == "test", "patient_id"])
    leaked = train_p & test_p
    print(f"Patients in BOTH train and test: {len(leaked)} of {len(test_p)} test patients")


def main():
    ap = argparse.ArgumentParser(description="Step 1: build manifest and splits")
    ap.add_argument("--data_root", required=True,
                    help="folder that contains Parasitized/ and Uninfected/ (searched recursively)")
    ap.add_argument("--verify", action="store_true", help="open every image and drop corrupted ones")
    args = ap.parse_args()

    data_root = Path(args.data_root)
    df = collect_images(data_root, verify=args.verify)

    print(f"Unique images : {len(df)}")
    print(df["class"].value_counts().to_string())
    n_pat = df.groupby("patient_id")["label"].max()
    print(f"Patients      : {len(n_pat)} ({int(n_pat.sum())} infected, {int((n_pat == 0).sum())} uninfected)")

    patient_df = patient_level_split(df)
    random_df = random_image_split(df)

    summarize(patient_df, "PATIENT-LEVEL split (used for all experiments)")
    summarize(random_df, "RANDOM image split (for comparison only)")

    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(SPLITS_DIR / "manifest.csv", index=False)
    patient_df.to_csv(SPLITS_DIR / "patient_split.csv", index=False)
    random_df.to_csv(SPLITS_DIR / "random_split.csv", index=False)
    print(f"\nSaved CSVs to {SPLITS_DIR}")


if __name__ == "__main__":
    main()

### `src/dataset.py` — Step 2: preprocessing and augmentation

In [ ]:
%%writefile src/dataset.py
"""
STEP 2 - DATASET AND PREPROCESSING
==================================
Loads cell images listed in a split CSV (from Step 1) and turns them into
tensors for training.

Preprocessing (identical on Kaggle, laptop and Raspberry Pi):
    1. Read image as RGB
    2. Resize to IMG_SIZE x IMG_SIZE (bilinear)
    3. Scale pixels to [0, 1]
    4. Channel order C x H x W
ImageNet mean/std normalization is done INSIDE the model (models.py), so the
Pi only has to do steps 1-4 with plain numpy - no torch needed on the Pi.

Augmentation (training only): flips, rotation, colour jitter. Cells have no
fixed orientation, and stain colour varies between slides and microscopes.
"""
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

from config import BATCH_SIZE, IMG_SIZE, NUM_WORKERS


# ----------------------------------------------------------------------
# Preprocessing shared with deployment
# ----------------------------------------------------------------------
def preprocess_pil(img: Image.Image, img_size: int = IMG_SIZE) -> np.ndarray:
    """PIL image -> float32 array (3, H, W) in [0, 1]. Same code runs on the Pi."""
    img = img.convert("RGB").resize((img_size, img_size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr.transpose(2, 0, 1)


def build_transforms(train: bool, img_size: int = IMG_SIZE):
    """torchvision pipeline. The eval pipeline matches preprocess_pil exactly."""
    if train:
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(20),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
            T.ToTensor(),
        ])
    return T.Compose([T.Resize((img_size, img_size)), T.ToTensor()])


# ----------------------------------------------------------------------
# Dataset
# ----------------------------------------------------------------------
class MalariaDataset(Dataset):
    """Returns (image_tensor, label). label: 0 = Uninfected, 1 = Parasitized."""

    def __init__(self, df: pd.DataFrame, data_root, transform):
        self.paths = df["path"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.root / self.paths[i]) as img:
            x = self.transform(img.convert("RGB"))
        return x, torch.tensor(self.labels[i], dtype=torch.long)


def load_split(split_csv) -> dict:
    """Read a split CSV and return {'train': df, 'val': df, 'test': df}."""
    df = pd.read_csv(split_csv)
    return {s: df[df["split"] == s].reset_index(drop=True) for s in ("train", "val", "test")}


def build_loaders(split_csv, data_root, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    """DataLoaders for train (shuffled, augmented) and val/test (plain)."""
    parts = load_split(split_csv)
    pin = torch.cuda.is_available()
    loaders = {}
    for name, df in parts.items():
        train = name == "train"
        ds = MalariaDataset(df, data_root, build_transforms(train))
        loaders[name] = DataLoader(ds, batch_size=batch_size, shuffle=train,
                                   num_workers=num_workers, pin_memory=pin,
                                   drop_last=False)
    return loaders, parts

### `src/models.py` — Step 3: model

In [ ]:
%%writefile src/models.py
"""
STEP 3 - MODEL
==============
A small ImageNet-pretrained CNN with a 2-class head.

EdgeClassifier wraps the backbone and adds ImageNet normalization as the
first operation, so the exported model takes plain [0, 1] RGB input.
Output: 2 logits [Uninfected, Parasitized].

Supported backbones (for the later architecture comparison):
    mobilenet_v3_small  (~1.5 M params, default)
    mobilenet_v2        (~2.2 M params)
    resnet18            (~11 M params)
"""
import io

import torch
import torch.nn as nn
import torchvision.models as tvm

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_backbone(arch: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    """Create a torchvision backbone and replace its last layer with a 2-class layer."""
    factories = {
        "mobilenet_v3_small": (tvm.mobilenet_v3_small, tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1),
        "mobilenet_v2": (tvm.mobilenet_v2, tvm.MobileNet_V2_Weights.IMAGENET1K_V1),
        "resnet18": (tvm.resnet18, tvm.ResNet18_Weights.IMAGENET1K_V1),
    }
    if arch not in factories:
        raise ValueError(f"Unknown arch '{arch}'. Choose from {list(factories)}")
    fn, weights = factories[arch]

    try:
        model = fn(weights=weights if pretrained else None)
        loaded = pretrained
    except Exception as e:          # e.g. no internet on Kaggle
        print(f"[warn] could not load pretrained weights ({e}); training from scratch")
        model = fn(weights=None)
        loaded = False

    if not loaded:
        # torchvision MobileNets use BatchNorm momentum 0.01, which makes the
        # running statistics converge very slowly when training from scratch
        # (eval predictions stay near-constant). 0.1 fixes that.
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.momentum = 0.1

    if arch.startswith("mobilenet"):
        last = model.classifier[-1]
        model.classifier[-1] = nn.Linear(last.in_features, num_classes)
    else:  # resnet
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


class EdgeClassifier(nn.Module):
    """Backbone + built-in normalization. Input: (N, 3, H, W) in [0, 1]."""

    def __init__(self, arch: str = "mobilenet_v3_small", pretrained: bool = True):
        super().__init__()
        self.arch = arch
        self.backbone = build_backbone(arch, 2, pretrained)
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, x):
        return self.backbone((x - self.mean) / self.std)


# ----------------------------------------------------------------------
# Model statistics (reported in the results table)
# ----------------------------------------------------------------------
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def model_size_mb(model: nn.Module) -> float:
    """Size of the saved FP32 weights in MB."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6

### `src/metrics.py` — Step 5: model metrics, temperature scaling, LED threshold

In [ ]:
%%writefile src/metrics.py
"""
STEP 5 - MODEL METRICS
======================
Everything is computed from P(Parasitized) and a decision threshold
(the LED turns on when P >= threshold).

Metrics
-------
accuracy      : fraction correct
sensitivity   : infected cells caught            TP / (TP + FN)   <- most important
specificity   : healthy cells correctly cleared  TN / (TN + FP)
precision     : LED-on events that are truly infected
f1            : balance of precision and sensitivity
auroc         : threshold-free ranking quality
ece           : expected calibration error - does a 90% confidence mean 90% correct?
brier         : mean squared error of the probabilities
"""
import numpy as np
import torch
from sklearn.metrics import roc_auc_score


def softmax_np(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    z = logits / temperature
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def expected_calibration_error(labels, prob_pos, n_bins: int = 15) -> float:
    """ECE on the confidence of the predicted class (standard definition)."""
    labels = np.asarray(labels)
    prob_pos = np.asarray(prob_pos)
    pred = (prob_pos >= 0.5).astype(int)
    conf = np.where(pred == 1, prob_pos, 1 - prob_pos)
    correct = (pred == labels).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0.5 else (conf >= lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def compute_metrics(labels, prob_pos, threshold: float = 0.5) -> dict:
    labels = np.asarray(labels).astype(int)
    prob_pos = np.asarray(prob_pos, dtype=np.float64)
    pred = (prob_pos >= threshold).astype(int)

    tp = int(((pred == 1) & (labels == 1)).sum())
    tn = int(((pred == 0) & (labels == 0)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum())
    fn = int(((pred == 0) & (labels == 1)).sum())
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * sens / max(prec + sens, 1e-12)
    try:
        auroc = roc_auc_score(labels, prob_pos)
    except ValueError:      # only one class present
        auroc = float("nan")

    return {
        "threshold": float(threshold),
        "accuracy": (tp + tn) / len(labels),
        "sensitivity": sens,
        "specificity": spec,
        "precision": prec,
        "f1": f1,
        "auroc": float(auroc),
        "ece": expected_calibration_error(labels, prob_pos),
        "brier": float(np.mean((prob_pos - labels) ** 2)),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def threshold_for_sensitivity(labels, prob_pos, target: float) -> float:
    """
    Highest threshold whose sensitivity is still >= target.
    (Highest = fewest false alarms while catching enough infections.)
    """
    labels = np.asarray(labels).astype(int)
    pos_scores = np.sort(np.asarray(prob_pos)[labels == 1])
    if len(pos_scores) == 0:
        return 0.5
    # allow at most floor((1 - target) * n_pos) positives below the threshold
    k = int(np.floor((1 - target) * len(pos_scores)))
    return float(pos_scores[k])


def fit_temperature(logits: np.ndarray, labels: np.ndarray, max_iter: int = 200) -> float:
    """
    Temperature scaling: find T > 0 that minimizes NLL of softmax(logits / T)
    on the validation set. One number; does not change which class wins.
    """
    x = torch.tensor(logits, dtype=torch.float64)
    y = torch.tensor(labels, dtype=torch.long)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=max_iter)

    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(x / log_t.exp(), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())

### `src/sysmon.py` — system metrics monitor (Kaggle and Pi)

In [ ]:
%%writefile src/sysmon.py
"""
SYSTEM METRICS MONITOR
======================
Samples hardware usage in a background thread while code runs.
Used during training on Kaggle AND during inference on the Raspberry Pi.

Recorded every `interval` seconds:
    cpu_percent      overall CPU usage (%)
    ram_used_mb      system RAM in use
    proc_rss_mb      RAM used by this Python process
    gpu_util / gpu_mem_mb   via nvidia-smi (Kaggle GPU only)
    cpu_temp_c       CPU temperature (Raspberry Pi / Linux thermal zone)

Usage
-----
    mon = SystemMonitor(interval=1.0)
    mon.start()
    ... work ...
    mon.stop()
    print(mon.summary())
"""
import os
import platform
import shutil
import subprocess
import threading
import time

import psutil


def read_cpu_temp():
    """CPU temperature in Celsius (works on Raspberry Pi); None if unavailable."""
    path = "/sys/class/thermal/thermal_zone0/temp"
    try:
        with open(path) as f:
            return int(f.read().strip()) / 1000.0
    except Exception:
        return None


def read_gpu():
    """(utilization %, memory used MB) of GPU 0 via nvidia-smi; (None, None) if no GPU."""
    if shutil.which("nvidia-smi") is None:
        return None, None
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
             "--format=csv,noheader,nounits", "-i", "0"],
            capture_output=True, text=True, timeout=2).stdout.strip()
        util, mem = out.split(",")
        return float(util), float(mem)
    except Exception:
        return None, None


def system_info() -> dict:
    """Static description of the machine (goes into the report)."""
    info = {
        "platform": platform.platform(),
        "machine": platform.machine(),
        "python": platform.python_version(),
        "cpu_cores_logical": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 2),
    }
    try:
        import torch
        info["torch"] = torch.__version__
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
            info["gpu_mem_total_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    except ImportError:
        pass
    try:
        import onnxruntime as ort
        info["onnxruntime"] = ort.__version__
    except ImportError:
        pass
    return info


class SystemMonitor:
    def __init__(self, interval: float = 1.0):
        self.interval = interval
        self.samples = []
        self._stop = threading.Event()
        self._thread = None
        self._proc = psutil.Process(os.getpid())

    def _sample(self):
        gpu_util, gpu_mem = read_gpu()
        return {
            "t": time.time(),
            "cpu_percent": psutil.cpu_percent(interval=None),
            "ram_used_mb": psutil.virtual_memory().used / 1e6,
            "proc_rss_mb": self._proc.memory_info().rss / 1e6,
            "gpu_util": gpu_util,
            "gpu_mem_mb": gpu_mem,
            "cpu_temp_c": read_cpu_temp(),
        }

    def _run(self):
        psutil.cpu_percent(interval=None)       # first call primes the counter
        while not self._stop.wait(self.interval):
            self.samples.append(self._sample())

    def start(self):
        self.samples = []
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join()
        if not self.samples:                     # very short runs: take one sample
            self.samples.append(self._sample())
        return self

    def summary(self) -> dict:
        """Mean and max of every recorded quantity."""
        out = {}
        keys = [k for k in self.samples[0] if k != "t"] if self.samples else []
        for k in keys:
            vals = [s[k] for s in self.samples if s[k] is not None]
            if vals:
                out[f"{k}_mean"] = sum(vals) / len(vals)
                out[f"{k}_max"] = max(vals)
        return out

### `src/train.py` — Step 4: training loop

In [ ]:
%%writefile src/train.py
"""
STEP 4 - TRAINING
=================
Fine-tunes the model on the training split and keeps the checkpoint with the
best validation AUROC.

Per epoch it logs
    model metrics : train loss, val loss, val accuracy / sensitivity / specificity / AUROC
    system metrics: epoch time, throughput (images/s), GPU util & memory,
                    peak GPU memory allocated by PyTorch, CPU %, process RAM

Outputs (in out_dir)
    best_model.pth   weights of the best epoch
    history.csv      one row per epoch with all metrics above

Usage (script)
    python src/train.py --data_root <path> --split_csv splits/patient_split.csv
"""
import argparse
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import config as C
from dataset import build_loaders
from metrics import compute_metrics, softmax_np
from models import EdgeClassifier
from sysmon import SystemMonitor


def set_seed(seed: int = C.SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict_logits(model, loader, device):
    """Run the model over a loader; return (logits [N,2], labels [N], mean loss)."""
    model.eval()
    crit = nn.CrossEntropyLoss(reduction="sum")
    all_logits, all_labels, total_loss = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            out = model(x)
        out = out.float()
        total_loss += crit(out, y).item()
        all_logits.append(out.cpu().numpy())
        all_labels.append(y.cpu().numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    return logits, labels, total_loss / len(labels)


def train_one_epoch(model, loader, optimizer, scaler, device):
    model.train()
    crit = nn.CrossEntropyLoss()
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            loss = crit(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(y)
        n += len(y)
    return total_loss / n, n


def fit(model, loaders, out_dir, epochs=C.EPOCHS, lr=C.LR, weight_decay=C.WEIGHT_DECAY):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

    history, best_auroc = [], -1.0
    print(f"Training {model.arch} on {device} for {epochs} epochs")

    for epoch in range(1, epochs + 1):
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        mon = SystemMonitor(interval=1.0).start()
        t0 = time.time()

        train_loss, n_seen = train_one_epoch(model, loaders["train"], optimizer, scaler, device)
        train_time = time.time() - t0
        val_logits, val_labels, val_loss = predict_logits(model, loaders["val"], device)
        epoch_time = time.time() - t0
        mon.stop()
        scheduler.step()

        m = compute_metrics(val_labels, softmax_np(val_logits)[:, 1])
        sysm = mon.summary()
        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": m["accuracy"],
            "val_sensitivity": m["sensitivity"],
            "val_specificity": m["specificity"],
            "val_auroc": m["auroc"],
            "epoch_time_s": epoch_time,
            "train_throughput_img_s": n_seen / train_time,
            "gpu_peak_alloc_mb": (torch.cuda.max_memory_allocated() / 1e6) if device.type == "cuda" else None,
            "gpu_util_mean": sysm.get("gpu_util_mean"),
            "gpu_mem_max_mb": sysm.get("gpu_mem_mb_max"),
            "cpu_percent_mean": sysm.get("cpu_percent_mean"),
            "proc_rss_max_mb": sysm.get("proc_rss_mb_max"),
        }
        history.append(row)

        flag = ""
        if m["auroc"] > best_auroc:
            best_auroc = m["auroc"]
            torch.save(model.state_dict(), out_dir / "best_model.pth")
            flag = "  <- best"
        print(f"ep {epoch:02d} | loss {train_loss:.4f}/{val_loss:.4f} | "
              f"acc {m['accuracy']:.4f} sens {m['sensitivity']:.4f} spec {m['specificity']:.4f} "
              f"auroc {m['auroc']:.4f} | {epoch_time:.0f}s {row['train_throughput_img_s']:.0f} img/s{flag}")

    hist = pd.DataFrame(history)
    hist.to_csv(out_dir / "history.csv", index=False)
    model.load_state_dict(torch.load(out_dir / "best_model.pth", map_location=device))
    return model, hist


def main():
    ap = argparse.ArgumentParser(description="Step 4: train")
    ap.add_argument("--data_root", required=True)
    ap.add_argument("--split_csv", default=str(C.SPLITS_DIR / "patient_split.csv"))
    ap.add_argument("--arch", default=C.ARCH)
    ap.add_argument("--epochs", type=int, default=C.EPOCHS)
    ap.add_argument("--out_dir", default=str(C.OUTPUT_DIR / C.ARCH))
    args = ap.parse_args()

    set_seed()
    loaders, _ = build_loaders(args.split_csv, args.data_root)
    model = EdgeClassifier(args.arch, pretrained=C.PRETRAINED)
    fit(model, loaders, args.out_dir, epochs=args.epochs)


if __name__ == "__main__":
    main()

### `src/export_onnx.py` — Step 7: ONNX export

In [ ]:
%%writefile src/export_onnx.py
"""
STEP 7 - EXPORT TO ONNX
=======================
Converts the trained PyTorch model to ONNX, the format the Raspberry Pi runs
with ONNX Runtime (no PyTorch needed on the Pi).

Input : "input"  float32 (N, 3, IMG_SIZE, IMG_SIZE), RGB in [0, 1]
Output: "logits" float32 (N, 2)  [Uninfected, Parasitized]

After export we check that ONNX and PyTorch give the same outputs.
"""
import numpy as np
import onnx
import onnxruntime as ort
import torch

from config import IMG_SIZE


def export_onnx(model, path, img_size: int = IMG_SIZE, opset: int = 17):
    model = model.cpu().eval()
    dummy = torch.rand(1, 3, img_size, img_size)
    kwargs = dict(input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
                  opset_version=opset)
    try:
        torch.onnx.export(model, dummy, str(path), dynamo=False, **kwargs)
    except TypeError:                     # older torch without the dynamo argument
        torch.onnx.export(model, dummy, str(path), **kwargs)
    onnx.checker.check_model(onnx.load(str(path)))
    return path


def check_parity(model, onnx_path, img_size: int = IMG_SIZE, n: int = 8) -> float:
    """Max absolute difference between PyTorch and ONNX logits on random inputs."""
    model = model.cpu().eval()
    x = torch.rand(n, 3, img_size, img_size)
    with torch.no_grad():
        ref = model(x).numpy()
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    out = sess.run(None, {"input": x.numpy()})[0]
    return float(np.abs(ref - out).max())

### `src/quantize.py` — Step 8: INT8 quantization

In [ ]:
%%writefile src/quantize.py
"""
STEP 8 - INT8 QUANTIZATION
==========================
Post-training STATIC quantization with ONNX Runtime.

    FP32 weights/activations  ->  INT8   (~4x smaller, faster on ARM CPUs)

Static = activation ranges are measured once on a small calibration set of
training images. (Dynamic quantization would leave the convolutions in FP32
on ONNX Runtime's CPU backend, giving almost no speed-up for a CNN.)

Format: QDQ (QuantizeLinear/DequantizeLinear nodes), per-channel weights.
"""
from pathlib import Path

import numpy as np
from PIL import Image
from onnxruntime.quantization import (CalibrationDataReader, CalibrationMethod,
                                      QuantFormat, QuantType, quantize_static)

from config import IMG_SIZE
from dataset import preprocess_pil


class ImageCalibrationReader(CalibrationDataReader):
    """Feeds calibration images one at a time to the quantizer."""

    def __init__(self, image_paths, img_size: int = IMG_SIZE):
        self.paths = list(image_paths)
        self.img_size = img_size
        self._it = iter(self.paths)

    def get_next(self):
        p = next(self._it, None)
        if p is None:
            return None
        with Image.open(p) as img:
            x = preprocess_pil(img, self.img_size)[None]      # (1, 3, H, W)
        return {"input": x}

    def rewind(self):
        self._it = iter(self.paths)


def quantize_int8(fp32_path, int8_path, calib_paths, method: str = "minmax"):
    fp32_path, int8_path = Path(fp32_path), Path(int8_path)

    # Recommended pre-processing (shape inference + graph optimization)
    prep_path = fp32_path.with_name(fp32_path.stem + "_prep.onnx")
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        quant_pre_process(str(fp32_path), str(prep_path))
        src = prep_path
    except Exception as e:
        print(f"[warn] quant_pre_process skipped ({e})")
        src = fp32_path

    calib_method = {"minmax": CalibrationMethod.MinMax,
                    "percentile": CalibrationMethod.Percentile,
                    "entropy": CalibrationMethod.Entropy}[method]

    quantize_static(
        model_input=str(src),
        model_output=str(int8_path),
        calibration_data_reader=ImageCalibrationReader(calib_paths),
        quant_format=QuantFormat.QDQ,
        per_channel=True,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QUInt8,
        calibrate_method=calib_method,
    )
    if prep_path.exists():
        prep_path.unlink()
    return int8_path

### `src/evaluate.py` — Step 9: ONNX evaluation and latency benchmark

In [ ]:
%%writefile src/evaluate.py
"""
STEP 9 - ONNX EVALUATION AND BENCHMARKING
=========================================
Runs an ONNX model (FP32 or INT8) the same way the Raspberry Pi will, and
measures both model metrics and system metrics.

    onnx_predict()      -> logits for a list of images (uses preprocess_pil)
    benchmark_latency() -> single-image latency (mean / p50 / p95 ms), FPS,
                           RAM used by the session, CPU usage, CPU temperature
    file_size_mb()      -> model file size on disk
"""
import os
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort
import psutil
from PIL import Image

from config import IMG_SIZE
from dataset import preprocess_pil
from sysmon import SystemMonitor


def make_session(onnx_path, threads: int = 4):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = threads
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(onnx_path), opts, providers=["CPUExecutionProvider"])


def file_size_mb(path) -> float:
    return os.path.getsize(path) / 1e6


def onnx_predict(session, image_paths, img_size: int = IMG_SIZE, batch: int = 64) -> np.ndarray:
    paths = list(image_paths)
    out = []
    for i in range(0, len(paths), batch):
        xs = []
        for p in paths[i:i + batch]:
            with Image.open(p) as img:
                xs.append(preprocess_pil(img, img_size))
        out.append(session.run(None, {"input": np.stack(xs)})[0])
    return np.concatenate(out)


def benchmark_latency(onnx_path, threads: int = 4, runs: int = 200, warmup: int = 20,
                      img_size: int = IMG_SIZE) -> dict:
    """Batch-1 latency, like one button press on the Pi."""
    proc = psutil.Process(os.getpid())
    rss_before = proc.memory_info().rss
    sess = make_session(onnx_path, threads)
    rss_after_load = proc.memory_info().rss

    x = np.random.rand(1, 3, img_size, img_size).astype(np.float32)
    for _ in range(warmup):
        sess.run(None, {"input": x})

    mon = SystemMonitor(interval=0.5).start()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        sess.run(None, {"input": x})
        times.append((time.perf_counter() - t0) * 1000)
    mon.stop()
    s = mon.summary()

    times = np.array(times)
    return {
        "model": Path(onnx_path).name,
        "threads": threads,
        "size_mb": file_size_mb(onnx_path),
        "latency_mean_ms": float(times.mean()),
        "latency_p50_ms": float(np.percentile(times, 50)),
        "latency_p95_ms": float(np.percentile(times, 95)),
        "fps": float(1000 / times.mean()),
        "session_ram_mb": (rss_after_load - rss_before) / 1e6,
        "cpu_percent_mean": s.get("cpu_percent_mean"),
        "cpu_temp_max_c": s.get("cpu_temp_c_max"),
    }

## 2. Imports and system information

In [ ]:
sys.path.insert(0, os.path.join(WORK, "src"))
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from PIL import Image

import config as C
from dataset import build_loaders, build_transforms, preprocess_pil
from models import EdgeClassifier, count_parameters, model_size_mb
from metrics import compute_metrics, softmax_np, fit_temperature, threshold_for_sensitivity, expected_calibration_error
from sysmon import system_info
from train import set_seed, fit, predict_logits
from export_onnx import export_onnx, check_parity
from quantize import quantize_int8
from evaluate import make_session, onnx_predict, benchmark_latency, file_size_mb

OUT = os.path.join(WORK, "outputs", C.ARCH)
DEPLOY = os.path.join(WORK, "outputs", "deploy")
os.makedirs(OUT, exist_ok=True); os.makedirs(DEPLOY, exist_ok=True)

SYS_TRAIN = system_info()
for k, v in SYS_TRAIN.items():
    print(f"{k:20s}: {v}")

## 3. Step 1 — Data preparation (patient-level split)

In [ ]:
!python src/data_prep.py --data_root "{DATA_ROOT}"

## 4. Step 2 — Dataset and preprocessing check

In [ ]:
set_seed()
SPLIT_CSV = os.path.join(WORK, "splits", "patient_split.csv")
loaders, parts = build_loaders(SPLIT_CSV, DATA_ROOT)
for k, df in parts.items():
    print(f"{k:5s}: {len(df):6d} images, {df.patient_id.nunique():3d} patients, "
          f"{(df.label == 1).mean():.1%} parasitized")

# Sample images
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for row, label in enumerate([1, 0]):
    sample = parts["train"][parts["train"].label == label].sample(6, random_state=0)
    for ax, p in zip(axes[row], sample.path):
        ax.imshow(Image.open(os.path.join(DATA_ROOT, p))); ax.axis("off")
    axes[row][0].set_title(C.CLASSES[label], loc="left")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "samples.png"), dpi=120); plt.show()

# The Pi uses preprocess_pil (numpy). Check it equals the torchvision eval transform.
p = os.path.join(DATA_ROOT, parts["test"].path[0])
with Image.open(p) as img:
    a = preprocess_pil(img)
    b = build_transforms(train=False)(img.convert("RGB")).numpy()
print("Max difference numpy vs torchvision preprocessing:", float(np.abs(a - b).max()))

## 5. Step 3 — Model

In [ ]:
model = EdgeClassifier(C.ARCH, pretrained=C.PRETRAINED)
print(f"Architecture : {C.ARCH}")
print(f"Parameters   : {count_parameters(model) / 1e6:.2f} M")
print(f"FP32 size    : {model_size_mb(model):.2f} MB")

## 6. Step 4 — Training (model metrics + system metrics per epoch)

In [ ]:
import time
t_start = time.time()
model, hist = fit(model, loaders, OUT, epochs=C.EPOCHS)
TOTAL_TRAIN_MIN = (time.time() - t_start) / 60
print(f"\nTotal training time: {TOTAL_TRAIN_MIN:.1f} min")
hist

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
e = hist.epoch
ax[0,0].plot(e, hist.train_loss, label="train"); ax[0,0].plot(e, hist.val_loss, label="val"); ax[0,0].set_title("Loss"); ax[0,0].legend()
for k in ["val_accuracy", "val_sensitivity", "val_specificity", "val_auroc"]:
    ax[0,1].plot(e, hist[k], label=k.replace("val_", ""))
ax[0,1].set_title("Validation metrics"); ax[0,1].legend()
ax[0,2].plot(e, hist.lr); ax[0,2].set_title("Learning rate")
ax[1,0].plot(e, hist.epoch_time_s); ax[1,0].set_title("Epoch time (s)")
ax[1,1].plot(e, hist.train_throughput_img_s); ax[1,1].set_title("Training throughput (img/s)")
if hist.gpu_util_mean.notna().any():
    ax[1,2].plot(e, hist.gpu_util_mean, label="GPU util %")
ax[1,2].plot(e, hist.cpu_percent_mean, label="CPU %"); ax[1,2].set_title("Hardware utilisation"); ax[1,2].legend()
for a in ax.flat: a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "training_curves.png"), dpi=120); plt.show()

## 7. Steps 5–6 — Test metrics, calibration and the LED threshold
* **Temperature scaling** (fit on validation) fixes over/under-confidence without changing predictions at 0.5.
* **LED threshold**: highest P(Parasitized) threshold that still catches ≥ `TARGET_SENSITIVITY` of infected cells on validation.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
val_logits, val_labels, _ = predict_logits(model, loaders["val"], device)
test_logits, test_labels, _ = predict_logits(model, loaders["test"], device)

T_FP32 = fit_temperature(val_logits, val_labels)
val_p = softmax_np(val_logits, T_FP32)[:, 1]
LED_THRESHOLD = threshold_for_sensitivity(val_labels, val_p, C.TARGET_SENSITIVITY)
print(f"Temperature T = {T_FP32:.3f}   LED threshold = {LED_THRESHOLD:.3f}")

p_raw = softmax_np(test_logits)[:, 1]
p_cal = softmax_np(test_logits, T_FP32)[:, 1]
torch_results = pd.DataFrame([
    {"setting": "raw, threshold 0.5", **compute_metrics(test_labels, p_raw, 0.5)},
    {"setting": "calibrated, threshold 0.5", **compute_metrics(test_labels, p_cal, 0.5)},
    {"setting": f"calibrated, LED threshold", **compute_metrics(test_labels, p_cal, LED_THRESHOLD)},
]).set_index("setting")
torch_results.round(4)

In [ ]:
from sklearn.metrics import roc_curve

def reliability(ax, labels, p, title):
    pred = (p >= 0.5).astype(int); conf = np.where(pred == 1, p, 1 - p); corr = (pred == labels)
    bins = np.linspace(0.5, 1, 11); mids, accs = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        if m.sum() > 0: mids.append(conf[m].mean()); accs.append(corr[m].mean())
    ax.plot([0.5, 1], [0.5, 1], "k--", lw=1); ax.plot(mids, accs, "o-")
    ax.set_title(f"{title}\nECE = {expected_calibration_error(labels, p):.4f}")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
r = torch_results.loc["calibrated, LED threshold"]
cm = np.array([[r.tn, r.fp], [r.fn, r.tp]])
ax[0].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm): ax[0].text(j, i, int(v), ha="center", va="center", fontsize=14)
ax[0].set_xticks([0, 1], ["LED off", "LED on"]); ax[0].set_yticks([0, 1], C.CLASSES); ax[0].set_title("Test confusion (LED threshold)")
fpr, tpr, _ = roc_curve(test_labels, p_cal)
ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], "k--", lw=1); ax[1].set_title(f"ROC (AUROC {r.auroc:.4f})"); ax[1].set_xlabel("1 - specificity"); ax[1].set_ylabel("sensitivity")
reliability(ax[2], test_labels, p_raw, "Reliability: raw")
reliability(ax[3], test_labels, p_cal, "Reliability: temperature-scaled")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "test_evaluation.png"), dpi=120); plt.show()

## 8. Step 7 — Export to ONNX

In [ ]:
FP32_ONNX = os.path.join(DEPLOY, "malaria_fp32.onnx")
export_onnx(model, FP32_ONNX)
print(f"Saved {FP32_ONNX}  ({file_size_mb(FP32_ONNX):.2f} MB)")
print("Max |PyTorch - ONNX| logit difference:", check_parity(model, FP32_ONNX))

## 9. Step 8 — INT8 static quantization

In [ ]:
INT8_ONNX = os.path.join(DEPLOY, "malaria_int8.onnx")
calib = parts["train"].sample(min(C.CALIB_IMAGES, len(parts["train"])), random_state=C.SEED)
calib_paths = [os.path.join(DATA_ROOT, p) for p in calib.path]
quantize_int8(FP32_ONNX, INT8_ONNX, calib_paths)
print(f"FP32: {file_size_mb(FP32_ONNX):.2f} MB  ->  INT8: {file_size_mb(INT8_ONNX):.2f} MB")

## 10. Step 9 — FP32 vs INT8: model metrics and system metrics
Both ONNX models run exactly as on the Pi (numpy preprocessing + ONNX Runtime on CPU).
INT8 is evaluated (a) with the FP32 temperature and LED threshold — what you get if you just deploy it —
and (b) re-calibrated with its own temperature.

In [ ]:
val_paths = [os.path.join(DATA_ROOT, p) for p in parts["val"].path]
test_paths = [os.path.join(DATA_ROOT, p) for p in parts["test"].path]
yv, yt = parts["val"].label.values, parts["test"].label.values

logits = {}
for name, path in [("FP32", FP32_ONNX), ("INT8", INT8_ONNX)]:
    sess = make_session(path, threads=os.cpu_count())
    logits[name] = (onnx_predict(sess, val_paths), onnx_predict(sess, test_paths))

T_INT8 = fit_temperature(logits["INT8"][0], yv)
rows = []
for name, T in [("FP32", T_FP32), ("INT8 (FP32 calibration)", T_FP32), ("INT8 (re-calibrated)", T_INT8)]:
    key = name.split()[0]
    p = softmax_np(logits[key][1], T)[:, 1]
    m = compute_metrics(yt, p, LED_THRESHOLD)
    m["ece_raw"] = expected_calibration_error(yt, softmax_np(logits[key][1])[:, 1])
    rows.append({"model": name, "temperature": T, **m})
onnx_results = pd.DataFrame(rows).set_index("model")

led_fp32 = softmax_np(logits["FP32"][1], T_FP32)[:, 1] >= LED_THRESHOLD
led_int8 = softmax_np(logits["INT8"][1], T_FP32)[:, 1] >= LED_THRESHOLD
print(f"T_FP32 = {T_FP32:.3f}, T_INT8 = {T_INT8:.3f}")
print(f"LED decisions that change after INT8: {(led_fp32 != led_int8).sum()} of {len(yt)}")
onnx_results[["temperature", "accuracy", "sensitivity", "specificity", "auroc", "ece_raw", "ece", "fn", "fp"]].round(4)

In [ ]:
# Latency / resources on this machine's CPU (reference only - the real numbers come from the Pi)
bench = pd.DataFrame([benchmark_latency(p, threads=t)
                      for p in (FP32_ONNX, INT8_ONNX) for t in (1, 4)])
bench.round(3)

## 11. Save results and deployment files

In [ ]:
deploy_config = {
    "model_fp32": "malaria_fp32.onnx",
    "model_int8": "malaria_int8.onnx",
    "classes": C.CLASSES,
    "positive_class_index": 1,
    "img_size": C.IMG_SIZE,
    "input": "RGB, resized to img_size x img_size (bilinear), float32 in [0,1], NCHW; normalization is inside the model",
    "temperature_fp32": T_FP32,
    "temperature_int8": T_INT8,
    "led_threshold": LED_THRESHOLD,
    "led_rule": "LED ON if softmax(logits / temperature)[1] >= led_threshold",
    "target_sensitivity": C.TARGET_SENSITIVITY,
    "arch": C.ARCH,
}
with open(os.path.join(DEPLOY, "deploy_config.json"), "w") as f:
    json.dump(deploy_config, f, indent=2)

# A few test images to copy to the Pi for the demo (balanced)
demo_dir = os.path.join(DEPLOY, "demo_images"); os.makedirs(demo_dir, exist_ok=True)
demo = pd.concat([parts["test"][parts["test"].label == l].sample(10, random_state=1) for l in (0, 1)])
for p in demo.path:
    shutil.copy(os.path.join(DATA_ROOT, p), os.path.join(demo_dir, os.path.basename(p)))
demo.assign(file=demo.path.map(os.path.basename))[["file", "class", "label", "patient_id"]].to_csv(
    os.path.join(demo_dir, "labels.csv"), index=False)

summary = {
    "system_training": SYS_TRAIN,
    "total_training_min": TOTAL_TRAIN_MIN,
    "params_m": count_parameters(model) / 1e6,
    "torch_test_metrics": torch_results.reset_index().to_dict("records"),
    "onnx_test_metrics": onnx_results.reset_index().to_dict("records"),
    "cpu_benchmark": bench.to_dict("records"),
}
with open(os.path.join(OUT, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)
onnx_results.to_csv(os.path.join(OUT, "onnx_results.csv"))
bench.to_csv(os.path.join(OUT, "cpu_benchmark.csv"), index=False)

shutil.make_archive("/kaggle/working/edai_outputs" if ON_KAGGLE else os.path.join(WORK, "edai_outputs"),
                    "zip", os.path.join(WORK, "outputs"))
print("Saved. Download edai_outputs.zip from the Output panel.")
for root, _, files in os.walk(DEPLOY):
    if "demo_images" in root: continue
    for fn in files: print(" ", os.path.join(root, fn))